# HR Analytics System — Employee Attrition Prediction

## 1. Introduction

This notebook is part of the **HR Analytics System** project, which combines
MySQL, Power BI, Python, Scikit-learn, and Streamlit to analyze employee data
and predict workforce attrition.

**Project Objective:**
Predict whether an employee is likely to leave the organization using
employee HR attributes from the IBM HR Employee Attrition dataset.

**Why Attrition Prediction Matters:**
Employee attrition is costly — it leads to recruitment expenses, loss of
institutional knowledge, and reduced team productivity. Identifying employees
at higher risk of leaving can help HR teams take proactive retention measures.

**What This Notebook Covers:**
1. Loading and exploring the HR dataset
2. Performing exploratory data analysis (EDA) to understand attrition patterns
3. Selecting relevant features for machine learning
4. Training and comparing multiple classification models
5. Evaluating model performance with appropriate metrics
6. Selecting and saving the final model for the Streamlit application

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

## 3. Load Dataset

The IBM HR Employee Attrition dataset contains approximately 1,470 employee
records with 35 attributes covering demographics, job characteristics,
satisfaction scores, compensation, and more.

In [ ]:
df = pd.read_csv("../data/raw/HR-Employee-Attrition.csv")

df.head()

## 4. Dataset Overview

Let's examine the basic structure of the dataset — its dimensions,
column names, data types, and the distribution of the target variable.

In [ ]:
# Dataset dimensions
print("Dataset Shape:", df.shape)
print(f"\nRows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

In [ ]:
# Column names
df.columns

In [ ]:
# Data types
print("Data Types:")
print(df.dtypes)

print("\nAttrition Distribution:")
print(df["Attrition"].value_counts())

print("\nAttrition Percentage:")
print((df["Attrition"].value_counts(normalize=True) * 100).round(2))

## 5. Data Quality Check

Before analysis, we check for missing values, duplicate rows, and
verify the target variable contains the expected values.

In [ ]:
# Missing values
print("Missing Values:")
print(df.isnull().sum().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nTarget Variable Unique Values:")
print(df["Attrition"].unique())

## 6. Exploratory Data Analysis (EDA)

This section explores patterns in the employee data that may be
associated with attrition. We examine both numerical and categorical
attributes to understand what differentiates employees who stayed
from those who left.

> **Note:** The patterns shown here are associations, not causal relationships.

### 6.1 Numerical Features Summary

In [ ]:
numeric_columns = [
    "Age",
    "DailyRate",
    "DistanceFromHome",
    "Education",
    "EnvironmentSatisfaction",
    "HourlyRate",
    "JobInvolvement",
    "JobLevel",
    "JobSatisfaction",
    "MonthlyIncome",
    "MonthlyRate",
    "NumCompaniesWorked",
    "PercentSalaryHike",
    "PerformanceRating",
    "RelationshipSatisfaction",
    "StockOptionLevel",
    "TotalWorkingYears",
    "TrainingTimesLastYear",
    "WorkLifeBalance",
    "YearsAtCompany",
    "YearsInCurrentRole",
    "YearsSinceLastPromotion",
    "YearsWithCurrManager"
]

numeric_summary = df[numeric_columns].describe().T

numeric_summary

### 6.2 Categorical Features Summary

In [ ]:
categorical_columns = [
    "BusinessTravel",
    "Department",
    "EducationField",
    "Gender",
    "JobRole",
    "MaritalStatus",
    "OverTime"
]

for column in categorical_columns:
    print(f"\n{'=' * 50}")
    print(column)
    print(f"{'=' * 50}")
    print(df[column].value_counts())
    print("\nPercentage:")
    print((df[column].value_counts(normalize=True) * 100).round(2))

### 6.3 Attrition by Department

In [ ]:
department_attrition = pd.crosstab(
    df["Department"],
    df["Attrition"]
)

department_attrition["Attrition_Rate"] = (
    department_attrition["Yes"]
    / department_attrition.sum(axis=1)
    * 100
).round(2)

department_attrition

### 6.4 Attrition by Job Role

In [ ]:
jobrole_attrition = pd.crosstab(
    df["JobRole"],
    df["Attrition"]
)

jobrole_attrition["Attrition_Rate"] = (
    jobrole_attrition["Yes"]
    / jobrole_attrition.sum(axis=1)
    * 100
).round(2)

jobrole_attrition.sort_values(
    "Attrition_Rate",
    ascending=False
)

### 6.5 Attrition by OverTime

In [ ]:
overtime_attrition = pd.crosstab(
    df["OverTime"],
    df["Attrition"]
)

overtime_attrition["Attrition_Rate"] = (
    overtime_attrition["Yes"]
    / overtime_attrition.sum(axis=1)
    * 100
).round(2)

overtime_attrition

### 6.6 Attrition by Job Satisfaction

In [ ]:
job_satisfaction_attrition = pd.crosstab(
    df["JobSatisfaction"],
    df["Attrition"]
)

job_satisfaction_attrition["Attrition_Rate"] = (
    job_satisfaction_attrition["Yes"]
    / job_satisfaction_attrition.sum(axis=1)
    * 100
).round(2)

job_satisfaction_attrition

### 6.7 Attrition by Age Group

In [ ]:
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[17, 25, 35, 45, 55, 60],
    labels=[
        "18-25",
        "26-35",
        "36-45",
        "46-55",
        "56-60"
    ]
)

age_attrition = pd.crosstab(
    df["AgeGroup"],
    df["Attrition"]
)

age_attrition["Attrition_Rate"] = (
    age_attrition["Yes"]
    / age_attrition.sum(axis=1)
    * 100
).round(2)

age_attrition

### 6.8 Attrition by Monthly Income

In [ ]:
income_attrition = pd.qcut(
    df["MonthlyIncome"],
    q=4,
    duplicates="drop"
)

income_attrition_table = pd.crosstab(
    income_attrition,
    df["Attrition"]
)

income_attrition_table["Attrition_Rate"] = (
    income_attrition_table["Yes"]
    / income_attrition_table.sum(axis=1)
    * 100
).round(2)

income_attrition_table

### 6.9 Attrition by Company Tenure

In [ ]:
df["CompanyTenureGroup"] = pd.cut(
    df["YearsAtCompany"],
    bins=[-1, 2, 5, 10, 20, 40],
    labels=[
        "0-2 years",
        "3-5 years",
        "6-10 years",
        "11-20 years",
        "21-40 years"
    ]
)

tenure_attrition = pd.crosstab(
    df["CompanyTenureGroup"],
    df["Attrition"]
)

tenure_attrition["Attrition_Rate"] = (
    tenure_attrition["Yes"]
    / tenure_attrition.sum(axis=1)
    * 100
).round(2)

tenure_attrition

### 6.10 Attrition by Business Travel

In [ ]:
business_travel_attrition = pd.crosstab(
    df["BusinessTravel"],
    df["Attrition"]
)

business_travel_attrition["Attrition_Rate"] = (
    business_travel_attrition["Yes"]
    / business_travel_attrition.sum(axis=1)
    * 100
).round(2)

business_travel_attrition

### 6.11 Attrition by Work-Life Balance

In [ ]:
worklife_attrition = pd.crosstab(
    df["WorkLifeBalance"],
    df["Attrition"]
)

worklife_attrition["Attrition_Rate"] = (
    worklife_attrition["Yes"]
    / worklife_attrition.sum(axis=1)
    * 100
).round(2)

worklife_attrition

### 6.12 Attrition by Job Level

In [ ]:
joblevel_attrition = pd.crosstab(
    df["JobLevel"],
    df["Attrition"]
)

joblevel_attrition["Attrition_Rate"] = (
    joblevel_attrition["Yes"]
    / joblevel_attrition.sum(axis=1)
    * 100
).round(2)

joblevel_attrition

### 6.13 Cleanup Temporary EDA Columns

In [ ]:
# Remove temporary columns created during EDA
df = df.drop(
    columns=["AgeGroup", "CompanyTenureGroup"],
    errors="ignore"
)

print("Temporary EDA columns removed.")
print("Dataset shape:", df.shape)

### 6.14 EDA Summary

In [ ]:
print("Final EDA Summary")
print("=" * 50)

print("Dataset Shape:", df.shape)

print("\nTotal Employees:", len(df))

print("\nAttrition Distribution:")
print(df["Attrition"].value_counts())

print("\nAttrition Percentage:")
print(
    (df["Attrition"].value_counts(normalize=True) * 100).round(2)
)

print("\nMissing Values:")
print(df.isnull().sum().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

---

## 7. Machine Learning Objective

The goal of the machine learning component is to build a classification model
that can estimate whether an employee is at higher risk of leaving the organization.

- **Target Variable:** `Attrition`
- **Positive Class (`Yes`):** Employee left the organization
- **Negative Class (`No`):** Employee stayed with the organization
- **Problem Type:** Binary Classification

The dataset is **imbalanced** — significantly more employees stayed than left.
This imbalance needs to be considered during model selection and evaluation.

The final trained model will be used by the Streamlit application to estimate
attrition risk for individual employees based on their HR attributes.

## 8. Feature Selection

Ten employee-related features are selected as input variables for the
prediction model. These features were chosen because they represent
demographic, job-related, financial, satisfaction, working-condition,
experience, and travel information.

In [ ]:
features = [
    "Age",
    "Department",
    "JobRole",
    "MonthlyIncome",
    "JobLevel",
    "OverTime",
    "JobSatisfaction",
    "WorkLifeBalance",
    "YearsAtCompany",
    "BusinessTravel"
]

X = df[features]
y = df["Attrition"]

print("Selected Features:")
print(features)

print("\nNumber of Features:", len(features))

print("\nFeature Data Shape:", X.shape)

print("\nTarget Data Shape:", y.shape)

## 9. Train-Test Split

The dataset is divided into training and testing sets.

- 80% of the data is used for model training.
- 20% is reserved for final model evaluation.
- Stratified splitting is used to maintain a similar Attrition distribution
  in both datasets.

The test set is kept separate so that the final model can be evaluated on
employee records that were not used during training.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

## 10. Feature Types

The selected features contain both numerical and categorical variables.

- **Numerical variables** contain measurable numeric values and will be
  standardized using `StandardScaler` before model training.
- **Categorical variables** contain groups or categories and will be converted
  into machine-readable numerical representations using `OneHotEncoder`.

In [ ]:
numeric_features = [
    "Age",
    "MonthlyIncome",
    "JobLevel",
    "JobSatisfaction",
    "WorkLifeBalance",
    "YearsAtCompany"
]

categorical_features = [
    "Department",
    "JobRole",
    "OverTime",
    "BusinessTravel"
]

print("Numerical Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)

## 11. Preprocessing Pipeline

A `ColumnTransformer` is used to apply the appropriate preprocessing to
each feature type:

- **Numerical features:** Standardized using `StandardScaler`
- **Categorical features:** Encoded using `OneHotEncoder`

This preprocessing is applied to the training data first, and then the
same transformation (without re-fitting) is applied to the test data.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Apply preprocessing
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Preprocessed training features shape:", X_train_processed.shape)
print("Preprocessed testing features shape:", X_test_processed.shape)

## 12. Model Training and Evaluation

Multiple classification models are trained and compared to find the best
approach for predicting employee attrition. Each model is evaluated using
the same test set for a fair comparison.

### 12.1 Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, confusion_matrix

# Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_processed, y_train)

# Predictions
lr_pred = lr_model.predict(X_test_processed)
lr_prob = lr_model.predict_proba(X_test_processed)[:, 1]

# Metrics
lr_accuracy = accuracy_score(y_test, lr_pred)
lr_precision = precision_score(y_test, lr_pred, pos_label="Yes")
lr_recall = recall_score(y_test, lr_pred, pos_label="Yes")
lr_f1 = f1_score(y_test, lr_pred, pos_label="Yes")
lr_roc_auc = roc_auc_score(y_test, lr_prob)

print("Logistic Regression Results")
print("-" * 30)
print(f"Accuracy : {lr_accuracy:.4f}")
print(f"Precision: {lr_precision:.4f}")
print(f"Recall   : {lr_recall:.4f}")
print(f"F1 Score : {lr_f1:.4f}")
print(f"ROC-AUC  : {lr_roc_auc:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, lr_pred))

### 12.2 Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Train Decision Tree
dt_model = DecisionTreeClassifier(random_state=42, max_depth=5)
dt_model.fit(X_train_processed, y_train)

# Predictions
dt_pred = dt_model.predict(X_test_processed)
dt_prob = dt_model.predict_proba(X_test_processed)[:, 1]

# Metrics
dt_accuracy = accuracy_score(y_test, dt_pred)
dt_precision = precision_score(y_test, dt_pred, pos_label="Yes")
dt_recall = recall_score(y_test, dt_pred, pos_label="Yes")
dt_f1 = f1_score(y_test, dt_pred, pos_label="Yes")
dt_roc_auc = roc_auc_score(y_test, dt_prob)

print("Decision Tree Results")
print("-" * 30)
print(f"Accuracy : {dt_accuracy:.4f}")
print(f"Precision: {dt_precision:.4f}")
print(f"Recall   : {dt_recall:.4f}")
print(f"F1 Score : {dt_f1:.4f}")
print(f"ROC-AUC  : {dt_roc_auc:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, dt_pred))

### 12.3 Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42
)
rf_model.fit(X_train_processed, y_train)

# Predictions
rf_pred = rf_model.predict(X_test_processed)
rf_prob = rf_model.predict_proba(X_test_processed)[:, 1]

# Metrics
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred, pos_label="Yes")
rf_recall = recall_score(y_test, rf_pred, pos_label="Yes")
rf_f1 = f1_score(y_test, rf_pred, pos_label="Yes")
rf_roc_auc = roc_auc_score(y_test, rf_prob)

print("Random Forest Results")
print("-" * 30)
print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1 Score : {rf_f1:.4f}")
print(f"ROC-AUC  : {rf_roc_auc:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

### 12.4 Balanced Random Forest

Since the target variable is imbalanced (more `No` than `Yes`), we use
`class_weight="balanced"` to give more weight to the minority class
during training. This helps the model better identify employees who may leave.

In [ ]:
# Train Balanced Random Forest
rf_balanced = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    class_weight="balanced",
    random_state=42
)
rf_balanced.fit(X_train_processed, y_train)

# Predictions
rf_balanced_pred = rf_balanced.predict(X_test_processed)
rf_balanced_prob = rf_balanced.predict_proba(X_test_processed)[:, 1]

# Metrics
rf_balanced_accuracy = accuracy_score(y_test, rf_balanced_pred)
rf_balanced_precision = precision_score(y_test, rf_balanced_pred, pos_label="Yes")
rf_balanced_recall = recall_score(y_test, rf_balanced_pred, pos_label="Yes")
rf_balanced_f1 = f1_score(y_test, rf_balanced_pred, pos_label="Yes")
rf_balanced_roc_auc = roc_auc_score(y_test, rf_balanced_prob)

print("Balanced Random Forest Results")
print("-" * 30)
print(f"Accuracy : {rf_balanced_accuracy:.4f}")
print(f"Precision: {rf_balanced_precision:.4f}")
print(f"Recall   : {rf_balanced_recall:.4f}")
print(f"F1 Score : {rf_balanced_f1:.4f}")
print(f"ROC-AUC  : {rf_balanced_roc_auc:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_balanced_pred))

## 13. Model Comparison

The following table compares all four models across key evaluation metrics.

**Understanding the metrics in the context of attrition prediction:**

- **Accuracy:** Overall percentage of correct predictions. Can be misleading
  with imbalanced data since predicting all employees as `No` would still
  yield high accuracy.
- **Precision:** Of all employees predicted to leave, how many actually left?
  Higher precision means fewer false alarms.
- **Recall:** Of all employees who actually left, how many did the model
  identify? Higher recall means fewer missed at-risk employees.
- **F1 Score:** Harmonic mean of precision and recall, useful when both
  matter.
- **ROC-AUC:** Measures the model's ability to distinguish between classes
  across all thresholds.

In [ ]:
# Model Comparison Table
comparison = {
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],

    "Logistic Regression": [
        lr_accuracy, lr_precision, lr_recall, lr_f1, lr_roc_auc
    ],

    "Decision Tree": [
        dt_accuracy, dt_precision, dt_recall, dt_f1, dt_roc_auc
    ],

    "Random Forest": [
        rf_accuracy, rf_precision, rf_recall, rf_f1, rf_roc_auc
    ],

    "Balanced Random Forest": [
        rf_balanced_accuracy, rf_balanced_precision,
        rf_balanced_recall, rf_balanced_f1, rf_balanced_roc_auc
    ]
}

comparison_df = pd.DataFrame(comparison)
comparison_df

## 14. Validation and Threshold Analysis

By default, a classification model uses a threshold of 0.50 — if the predicted
probability of attrition is >= 0.50, the employee is classified as `Yes`.

Changing this threshold affects the trade-off between precision and recall:

- **Lower threshold:** More employees are flagged as at-risk (higher recall),
  but more false positives (lower precision).
- **Higher threshold:** Fewer employees are flagged (higher precision),
  but more at-risk employees may be missed (lower recall).

To evaluate different thresholds, we create a validation set from the
training data and test multiple threshold values.

In [ ]:
from sklearn.model_selection import train_test_split

X_train_inner, X_val, y_train_inner, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

print("Training data:", X_train_inner.shape)
print("Validation data:", X_val.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

validation_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

X_train_inner_processed = validation_preprocessor.fit_transform(X_train_inner)
X_val_processed = validation_preprocessor.transform(X_val)

print("Training features:", X_train_inner_processed.shape)
print("Validation features:", X_val_processed.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_validation = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    class_weight="balanced",
    random_state=42
)

rf_validation.fit(
    X_train_inner_processed,
    y_train_inner
)

print("Validation Random Forest trained successfully!")

In [ ]:
rf_val_prob = rf_validation.predict_proba(X_val_processed)[:, 1]

print(rf_val_prob[:10])

### Threshold Analysis Results

Testing different probability thresholds on the validation set:

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.10, 0.61, 0.05)

for threshold in thresholds:
    y_val_threshold = np.where(
        rf_val_prob >= threshold,
        "Yes",
        "No"
    )

    precision = precision_score(
        y_val,
        y_val_threshold,
        pos_label="Yes"
    )

    recall = recall_score(
        y_val,
        y_val_threshold,
        pos_label="Yes"
    )

    f1 = f1_score(
        y_val,
        y_val_threshold,
        pos_label="Yes"
    )

    print(
        f"Threshold: {threshold:.2f} | "
        f"Precision: {precision:.2f} | "
        f"Recall: {recall:.2f} | "
        f"F1: {f1:.2f}"
    )

## 15. Final Model

Based on the model comparison, the **Balanced Random Forest** is selected
as the final model because:

- It has significantly better **recall** than other models, meaning it
  identifies more employees who actually left.
- The `class_weight="balanced"` parameter compensates for the imbalanced
  target variable.
- While accuracy is slightly lower, the trade-off is acceptable for an
  attrition risk estimation tool.

The final model is retrained on the full training set (not the validation
subset) using a probability threshold of 0.50.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

final_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    class_weight="balanced",
    random_state=42
)

final_rf.fit(
    X_train_processed,
    y_train
)

print("Final Balanced Random Forest trained successfully!")

## 16. Final Model Evaluation

Evaluating the final Balanced Random Forest model on the held-out test set.

In [ ]:
# Predict attrition probabilities on the final test set
final_test_prob = final_rf.predict_proba(X_test_processed)[:, 1]

# Apply our selected threshold
final_test_pred = np.where(
    final_test_prob >= 0.50,
    "Yes",
    "No"
)

print(final_test_pred[:10])

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

accuracy = accuracy_score(y_test, final_test_pred)
precision = precision_score(y_test, final_test_pred, pos_label="Yes")
recall = recall_score(y_test, final_test_pred, pos_label="Yes")
f1 = f1_score(y_test, final_test_pred, pos_label="Yes")
roc_auc = roc_auc_score(
    (y_test == "Yes").astype(int),
    final_test_prob
)

print("Final Model Results")
print("-------------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, final_test_pred))

## 17. Final Pipeline

For deployment in the Streamlit application, the preprocessing and model
are combined into a single `Pipeline` object. This ensures that any new
employee data passes through the same preprocessing steps before prediction.

The pipeline is trained on the original training data and verified on the
test set to confirm it produces the same results.

In [ ]:
from sklearn.pipeline import Pipeline

final_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=6,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

final_pipeline.fit(X_train, y_train)

print("Final ML pipeline trained successfully!")

In [ ]:
# Predict probabilities using the complete pipeline
pipeline_test_prob = final_pipeline.predict_proba(X_test)[:, 1]

# Apply our selected threshold
pipeline_test_pred = np.where(
    pipeline_test_prob >= 0.50,
    "Yes",
    "No"
)

print("First 10 probabilities:")
print(pipeline_test_prob[:10])

print("\nFirst 10 predictions:")
print(pipeline_test_pred[:10])

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

pipeline_accuracy = accuracy_score(y_test, pipeline_test_pred)
pipeline_precision = precision_score(
    y_test,
    pipeline_test_pred,
    pos_label="Yes"
)
pipeline_recall = recall_score(
    y_test,
    pipeline_test_pred,
    pos_label="Yes"
)
pipeline_f1 = f1_score(
    y_test,
    pipeline_test_pred,
    pos_label="Yes"
)
pipeline_roc_auc = roc_auc_score(
    (y_test == "Yes").astype(int),
    pipeline_test_prob
)

print("Pipeline Model Results")
print("----------------------")
print(f"Accuracy : {pipeline_accuracy:.4f}")
print(f"Precision: {pipeline_precision:.4f}")
print(f"Recall   : {pipeline_recall:.4f}")
print(f"F1 Score : {pipeline_f1:.4f}")
print(f"ROC-AUC  : {pipeline_roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pipeline_test_pred))

## 18. Model Interpretation — Feature Importance

The Random Forest model provides feature importance scores that indicate
how useful each feature was for making predictions.

> **Important:** Feature importance indicates which features the model
> found useful for prediction. It does **not** prove that a feature
> causes employees to leave.

In [ ]:
# Extract feature names after one-hot encoding
feature_names = (
    numeric_features
    + list(final_pipeline.named_steps["preprocessor"]
           .named_transformers_["cat"]
           .get_feature_names_out(categorical_features))
)

# Get feature importances from the model inside the pipeline
importances = final_pipeline.named_steps["model"].feature_importances_

# Create a sorted DataFrame
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

print("Feature Importance (Top 15):")
print("=" * 40)
for _, row in importance_df.head(15).iterrows():
    print(f"{row['Feature']:30s} {row['Importance']:.4f}")

## 19. Save Final Model

The final trained pipeline is saved as a `.pkl` file for use by the
Streamlit application.

**Saved model path:** `../python/attrition_model.pkl`

This is the production model used by the HR Analytics Streamlit application
to estimate employee attrition risk.

In [ ]:
import joblib

joblib.dump(
    final_pipeline,
    "../python/attrition_model.pkl"
)

print("Model saved successfully!")

## 20. Final ML Conclusions

The machine learning analysis was performed to estimate employee attrition risk
using selected demographic, job-related, satisfaction, compensation, and
work-related features.

### Models Tested

Several classification models were trained and compared:

| Model | Purpose |
|---|---|
| Logistic Regression | Simple baseline model |
| Decision Tree | Interpretable tree-based model |
| Random Forest | Ensemble model for better generalization |
| Balanced Random Forest | Handles class imbalance |

### Final Model Selection

Because the target variable is imbalanced, with significantly more employees
staying than leaving, the **Balanced Random Forest** model was selected for the
final prediction pipeline. The balanced approach improved the model's ability
to identify employees who may leave.

The final model was evaluated on an untouched test set using a probability
threshold of 0.50.

### Final Model Performance

- Accuracy: ~80.61%
- Precision: ~41.67%
- Recall: ~53.19%
- F1 Score: ~46.73%
- ROC-AUC: ~74.76%

The model's recall of ~53.19% indicates that it identified a substantial
portion of the employees who actually left in the test dataset.

### Deployment

The final preprocessing and Balanced Random Forest model were combined into a
single pipeline and saved as: `python/attrition_model.pkl`

This trained pipeline is used by the Streamlit application to estimate
employee attrition risk from employee information entered by the user.

### Limitations

- The model provides **risk estimates**, not definitive predictions of
  employee behavior.
- Feature importance shows what the model found useful for prediction,
  **not** what causes attrition.
- The model is trained on the IBM HR dataset and may not generalize directly
  to other organizations without retraining.
- This tool should **support** HR analysis and decision-making, not
  automatically make employment decisions.

Overall, the machine learning component demonstrates how historical HR data
can be transformed into a predictive analytics workflow and integrated into
an interactive HR analytics system.